In [1]:
import os
import sys
import pprint
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import McpWorkbench, StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [ ]:
model_client = load_model_client("lmstudio")

In [3]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-oss', 'host': 'http://localhost:11434', 'follow_redirects': True}
create_args: {'model': 'gpt-oss'}


In [4]:
PyMCP_FS = StdioServerParams(command="/run/media/jun/Jun_6T/Code/mcp_servers/PyMCP-FS/.venv/bin/python", args=["/run/media/jun/Jun_6T/Code/mcp_servers/PyMCP-FS/main.py","-d","/run/media/jun/Jun_6T/Code/TFM_BA_GPT/",	"/tmp/","--log-level","DEBUG","--log-file",	"mcp_debug.log"])


Mermaid_MCP = StdioServerParams(command="/run/media/jun/Jun_6T/Code/mcp_servers/Mermaid-MCP/venv/bin/python", args=[
				"/run/media/jun/Jun_6T/Code/mcp_servers/Mermaid-MCP/Mermaid-MCP.py",
				"-d",
				"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/",
				"-d",
				"/tmp/"])

PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)

In [5]:

# ---- terminate_agent system message (MOVE + only SVG) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  conversation log: /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl
  spec:             /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/functional_specification.md
  mermaid:          /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/process_diagram.mmd

**GOAL**
  Create: /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.svg

**STEPS STRICT ORDER**
  1 Read /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl and extract the first available "session_id" field value.
      If multiple lines: pick the first non-empty "session_id".
      If not found: set SESSION_ID = "sess-unknown".

2 Create folder /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/

  3) MOVE the three source files into that folder (do not copy):
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl             to /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/interview.jsonl
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/functional_specification.md to /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/functional_specification.md
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/process_diagram.mmd         to /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd
      Use the 'move_file' tool.    

4 Clean the Mermaid file **in place** (auto-strip code fences, normalize init JSON, tidy whitespace):
    clean_mermaid(
      input_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd",
      output_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.svg",
    overwrite=True
    )
    Report any issues returned by the tool (severity/code/message/recommendation).

5 Render Mermaid **SVG only** in one call:
      render_mermaid_svg(
      input_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd",
      output_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.svg",
      theme="forest"
      )

6 Confirm that /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.svg exists.

7 When everything is present, output exactly: **TERMINATE**

**RULES**
  Be explicit about each steps success or failure.
  If a tool call fails, retry up to 2 times with the same arguments; if still failing, state the failure and continue with remaining steps when possible.
"""


In [6]:
termination = TextMentionTermination("TERMINATE")


terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='0', arguments='{"path": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')]
---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"_meta\\": {\\"session_id\\": \\"sess-20231011162620\\", \\"ts\\": \\"2025-08-09T20:35:33.157973Z\\", \\"n_records\\": 2}}\\n{\\"speaker\\": \\"interviewer\\", \\"message\\": \\"QUESTION: Can you briefly describe the new product and its primary purpose?\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-08-09T20:35:33.157947Z\\"}\\n{\\"speaker\\": \\"user\\", \\"message\\": \\"The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer u

TaskResult(messages=[TextMessage(id='d35240cf-5103-4d12-90ed-f86da06f5853', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 11, 12, 19, 59, 475808, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='30bb9c3a-852d-4ea5-9d54-d70138fb7428', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=2530, completion_tokens=220), metadata={}, created_at=datetime.datetime(2025, 8, 11, 12, 20, 2, 324909, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='0', arguments='{"path": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='07532f51-3e75-447a-8b83-e47ef9088380', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 11, 12, 20, 2, 563547, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='[{"type": "text", "text"